# Checkout A/B test analysis

The experimental unit is one assigned user. **Conversion rate is the primary metric**: converted users divided by assigned users in each arm.

- **H0:** Control and Variant conversion rates are equal. **H1:** They differ.
- Use a **two-sided, two-proportion Z-test** with alpha = 0.05. Report the Variant-minus-Control effect and its 95% Newcombe confidence interval. The test relies on independent user assignments and sufficiently many conversions and nonconversions in each arm.
- **AOV is secondary:** mean checkout amount among converted users. Compare purchaser means with a two-sided Welch t-test and its 95% confidence interval. Its p-value is unadjusted; the primary conversion test determines the experiment conclusion.
- **Revenue per assigned user** includes zero revenue for nonconverters and is reported descriptively, without another hypothesis test.

AOV conditions on conversion, which treatment can change. The purchaser groups can therefore differ; an AOV difference does not prove that the same customers would spend more under one variant.


In [1]:
import pandas as pd
from IPython.display import Markdown, display
from scipy import stats
from statsmodels.stats.proportion import confint_proportions_2indep, proportions_ztest

from experiment_decision import ALPHA, conversion_conclusion
from validate_data import validate_experiment_data

# The export is one row per assigned user.
df = pd.read_csv("clean_data_for_stats.csv")
validate_experiment_data(df)

arm = df.groupby("variant").agg(
    users=("user_id", "size"),
    conversions=("converted", "sum"),
    total_revenue=("checkout_amount", "sum"),
)
control = arm.loc["Control"]
variant = arm.loc["Variant"]
control_rate = control.conversions / control.users
variant_rate = variant.conversions / variant.users
conversion_difference = variant_rate - control_rate
relative_lift = conversion_difference / control_rate

# Variant comes first, so the test statistic and interval use Variant minus Control.
z_stat, p_value_conversion = proportions_ztest(
    [variant.conversions, control.conversions],
    [variant.users, control.users],
    alternative="two-sided",
)
conversion_ci = confint_proportions_2indep(
    int(variant.conversions), int(variant.users),
    int(control.conversions), int(control.users),
    compare="diff", method="newcomb", alpha=ALPHA,
)

purchasers = df.loc[df["converted"] == 1]
control_amounts = purchasers.loc[purchasers["variant"] == "Control", "checkout_amount"]
variant_amounts = purchasers.loc[purchasers["variant"] == "Variant", "checkout_amount"]
control_aov = control_amounts.mean()
variant_aov = variant_amounts.mean()
aov_difference = variant_aov - control_aov
welch = stats.ttest_ind(variant_amounts, control_amounts, equal_var=False, alternative="two-sided")
aov_ci = welch.confidence_interval(confidence_level=1 - ALPHA)

control_revenue_per_user = control.total_revenue / control.users
variant_revenue_per_user = variant.total_revenue / variant.users
revenue_per_user_difference = variant_revenue_per_user - control_revenue_per_user

arm_results = pd.DataFrame(
    {
        "Assigned users": [f"{control.users:,.0f}", f"{variant.users:,.0f}"],
        "Conversions / purchasers": [f"{control.conversions:,.0f}", f"{variant.conversions:,.0f}"],
        "Conversion rate": [f"{control_rate:.2%}", f"{variant_rate:.2%}"],
        "Total revenue": [f"${control.total_revenue:,.2f}", f"${variant.total_revenue:,.2f}"],
        "AOV": [f"${control_aov:.2f}", f"${variant_aov:.2f}"],
        "Revenue / assigned user": [f"${control_revenue_per_user:.2f}", f"${variant_revenue_per_user:.2f}"],
    },
    index=["Control", "Variant"],
)
arm_results.index.name = "Arm"

effect_results = pd.DataFrame(
    [
        {
            "Metric": "Conversion rate (primary)",
            "Variant minus Control": f"{conversion_difference * 100:+.2f} pp",
            "Relative lift": f"{relative_lift:+.2%}",
            "95% CI": f"{conversion_ci[0] * 100:+.2f} to {conversion_ci[1] * 100:+.2f} pp",
            "Statistic": f"z = {z_stat:+.2f}",
            "Two-sided p": f"{p_value_conversion:.2e}",
        },
        {
            "Metric": "AOV (secondary)",
            "Variant minus Control": f"{aov_difference:+.2f} USD",
            "Relative lift": "n/a",
            "95% CI": f"{aov_ci.low:+.2f} to {aov_ci.high:+.2f} USD",
            "Statistic": f"t = {welch.statistic:+.2f}",
            "Two-sided p": f"{welch.pvalue:.2e}",
        },
        {
            "Metric": "Revenue / assigned user (descriptive)",
            "Variant minus Control": f"{revenue_per_user_difference:+.2f} USD",
            "Relative lift": "n/a",
            "95% CI": "n/a",
            "Statistic": "n/a",
            "Two-sided p": "n/a",
        },
    ]
).set_index("Metric")

display(Markdown("## Results"))
display(arm_results)
display(effect_results)
display(Markdown(f"**Primary conclusion (alpha = {ALPHA:.2f}):** {conversion_conclusion(p_value_conversion, conversion_difference)}"))


## Results

,Assigned users,Conversions / purchasers,Conversion rate,Total revenue,AOV,Revenue / assigned user
Arm,,,,,,
Control,"25,053","3,000",11.97%,"$151,652.02",$50.55,$6.05
Variant,"24,947","3,564",14.29%,"$196,194.21",$55.05,$7.86


,Variant minus Control,Relative lift,95% CI,Statistic,Two-sided p
Metric,,,,,
Conversion rate (primary),+2.31 pp,+19.30%,+1.72 to +2.90 pp,z = +7.65,1.96e-14
AOV (secondary),+4.50 USD,n/a,+3.76 to +5.23 USD,t = +11.99,9.31e-33
Revenue / assigned user (descriptive),+1.81 USD,n/a,n/a,n/a,n/a


**Primary conclusion (alpha = 0.05):** Variant has a statistically significant higher conversion rate.